# Projeto Avaliativo — Machine Learning e Visão Computacional
## E-commerce Churn: previsão de abandono de clientes

Este projeto implementa um **Pipeline Preditivo completo** para identificar clientes com maior probabilidade de abandonar uma plataforma de e-commerce (`Churn = 1`), seguindo as seis fases técnicas exigidas na Situação de Aprendizagem.

**Base utilizada:** `E Commerce Dataset - E Comm.csv`

### Objetivos
- realizar Análise Exploratória de Dados (EDA);
- tratar duplicidades, valores nulos e outliers;
- criar a variável calculada `cashback_por_pedido`;
- separar treino e teste de forma estratificada;
- balancear **somente o conjunto de treino**;
- aplicar escalonamento somente no KNN;
- testar diferentes valores de `n_neighbors` e `max_depth`;
- diagnosticar overfitting comparando treino e teste;
- comparar KNN e Árvore de Decisão pelas métricas e matrizes de confusão;
- transformar os resultados em um **veredito de negócio**;
- gerar automaticamente um `README.md` completo ao final.

> **Regra metodológica:** nenhuma informação do conjunto de teste será usada para ajustar imputadores, limites de outliers, encoder, scaler ou SMOTE.

> **Versão histórica — Fase 1 — EDA.**
> Esta versão contém somente o que foi desenvolvido até esta fase.


## 0. Configuração do ambiente

Esta célula instala a biblioteca necessária para balanceamento de classes.

In [ ]:
!pip -q install imbalanced-learn

## 1. Importação das bibliotecas

Serão utilizadas bibliotecas de análise de dados, visualização, pré-processamento, balanceamento e modelagem.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
RANDOM_STATE = 42

## 2. Carregamento do dataset

No Google Colab, o usuário fará o upload do CSV. O código identifica automaticamente o primeiro arquivo `.csv` enviado.

In [ ]:
uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Nenhum arquivo CSV foi enviado.")

CSV_NAME = csv_files[0]
df_raw = pd.read_csv(CSV_NAME)

print(f"Arquivo carregado: {CSV_NAME}")
print(f"Dimensões iniciais: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas")
display(df_raw.head())

# Fase 1 — Análise Exploratória de Dados (EDA)

A EDA documenta dimensões, tipos, estatísticas, distribuição do alvo, distribuições numéricas e correlações. As observações obtidas nesta etapa orientarão as decisões de preparação.

In [ ]:
df = df_raw.copy()

print("Dimensões:", df.shape)
print("\nTipos de dados:")
display(df.dtypes.to_frame("dtype"))

print("\nValores nulos por coluna:")
display(df.isna().sum().to_frame("nulos").sort_values("nulos", ascending=False))

print("\nResumo estatístico:")
display(df.describe(include="all").T)

### EDA — Distribuição da variável alvo

O gráfico verifica visualmente o desbalanceamento entre clientes que permaneceram e clientes que abandonaram a plataforma.

In [ ]:
target_counts = df["Churn"].value_counts().sort_index()
target_pct = df["Churn"].value_counts(normalize=True).sort_index() * 100

plt.figure(figsize=(7, 4))
bars = plt.bar(["Não Churn (0)", "Churn (1)"], target_counts.values)
plt.title("Distribuição da variável alvo — Churn")
plt.ylabel("Quantidade de clientes")
plt.xlabel("Classe")

for bar, value, pct in zip(bars, target_counts.values, target_pct.values):
    plt.text(
        bar.get_x() + bar.get_width()/2, value,
        f"{value}\n({pct:.1f}%)",
        ha="center", va="bottom"
    )

plt.tight_layout()
plt.show()

### EDA — Distribuições numéricas

Os histogramas permitem observar assimetria, concentração e possíveis valores extremos nas principais variáveis quantitativas.

In [ ]:
numeric_eda = [
    "Tenure", "WarehouseToHome", "HourSpendOnApp",
    "OrderAmountHikeFromlastYear", "OrderCount",
    "DaySinceLastOrder", "CashbackAmount"
]

for col in numeric_eda:
    plt.figure(figsize=(7, 4))
    plt.hist(df[col].dropna(), bins=30)
    plt.title(f"Distribuição — {col}")
    plt.xlabel(col)
    plt.ylabel("Frequência")
    plt.tight_layout()
    plt.show()

### EDA — Correlação de Pearson

A matriz de correlação considera as variáveis numéricas e auxilia na identificação de relações lineares e possíveis redundâncias.

In [ ]:
corr = df.select_dtypes(include=np.number).corr(method="pearson")

plt.figure(figsize=(11, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlação de Pearson")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Mapa de calor — Correlação de Pearson")
plt.tight_layout()
plt.show()

### EDA — Interpretação e tomada de decisão

A célula seguinte transforma os resultados observados em uma síntese textual.

In [ ]:
churn_rate = df["Churn"].mean() * 100
missing_cols = df.columns[df.isna().any()].tolist()

print(
    f"A base possui {df.shape[0]:,} registros e {df.shape[1]} variáveis. "
    f"A taxa de churn observada é de {churn_rate:.2f}%, indicando uma classe minoritária "
    f"que deve ser considerada durante a modelagem. Foram encontradas {len(missing_cols)} "
    f"colunas com valores ausentes. A preparação deverá tratar esses nulos sem utilizar "
    f"informações do conjunto de teste. A presença de valores extremos será avaliada por "
    f"IQR, com atenção especial ao KNN, que utiliza distâncias, enquanto a Árvore de "
    f"Decisão é menos sensível à escala das variáveis."
)